## 配置


### amd gpu


In [1]:
!sudo pacman -S --needed --noconfirm rocm-opencl-runtime rocm-hip-runtime


 there is nothing to do


### install


In [ ]:
%%bash
path_append() {
	local tgt=${1:-}
	[[ -z "$tgt" ]] && return 1

	local e_l="export PATH=\"$tgt:\$PATH\""
	if grep -qF "$e_l" "$HOME/.zshrc"; then
		echo "  ✓ PATH already contains $tgt"
	else
		echo "$e_l" >>"$HOME/.zshrc"
		echo "  ✓ Added $tgt to PATH"
	fi
}
install_ollama(){
    local install_dir=${1:-/data/.path/.ollama}
    [[ -d "$install_dir" ]] && return
    cd ~/Downloads
    # 检查文件是否存在，不存在才下载
    # aria2 专用代理参数
    proxy_opts="--all-proxy=https://192.168.0.103:7897"
    
    # 使用 aria2 下载，避免重复下载
    [[ ! -f "ollama-linux-amd64.tgz" ]] && \
        aria2c -x 16 -s 16 --continue=true $proxy_opts \
        https://ollama.com/download/ollama-linux-amd64.tgz
    
    [[ ! -f "ollama-linux-amd64-rocm.tgz" ]] && \
        aria2c -x 16 -s 16 --continue=true $proxy_opts \
        https://ollama.com/download/ollama-linux-amd64-rocm.tgz
    # 解压文件
    mkdir -p "$install_dir"
    tar -zxf ollama-linux-amd64.tgz -C "$install_dir"
    tar -zxf ollama-linux-amd64-rocm.tgz -C "$install_dir"

    # 检查组是否存在，不存在才创建
    getent group ollama >/dev/null || sudo groupadd ollama
    groups $USER | grep -q ollama || sudo usermod -aG ollama $USER
    
    rm -rf $HOME/.ollama
    ln -sf "$install_dir" "$HOME/.ollama"
    # 添加环境变量
    path_append "$install_dir/bin" 
    
    echo "  ✓ ollama installed"
    echo "source $HOME/.zshrc"
    
}

install_ollama


### start serve


In [ ]:
%%bash
## run
HSA_OVERRIDE_GFX_VERSION=10.3.0 ollama serve


### pull model


In [ ]:
ollama pull ollama3.2:3b


### run model in shell


In [ ]:
%%bash
ollama list
ollama run ollama3.2:3b


## MCP Bridge 集成


### 安装 ollama-mcp-bridge


In [ ]:
# >>>>>>>>>>>>>>>>>>>>安装 ollama-mcp-bridge>>>>>>>>>>>>>>>
%pip install --upgrade ollama-mcp-bridge


### 配置 MCP 服务器


In [ ]:
%%bash
# >>>>>>>>>>>>>>>>>>>>创建 MCP 配置文件>>>>>>>>>>>>>>>
cat > mcp-config.json << 'EOF'
{
  "mcpServers": {
    "filesystem": {
      "command": "npx",
      "args": ["-y", "@modelcontextprotocol/server-filesystem", "/tmp"]
    }
  }
}
EOF


### 启动 MCP Bridge


In [ ]:
%%bash
# >>>>>>>>>>>>>>>>>>>>启动 bridge (在新终端运行)>>>>>>>>>>>>>>>
ollama-mcp-bridge --config /data/.manjaro/manjaro_usage/.amazonq/agents/default.json --host 0.0.0.0 --port 8009


### Python 调用 MCP 工具


In [19]:
import ollama

# >>>>>>>>>>>>>>>>>>>>配置客户端指向 bridge>>>>>>>>>>>>>>>
client = ollama.Client(host='http://localhost:8008')

# >>>>>>>>>>>>>>>>>>>>调用时自动使用 MCP 工具>>>>>>>>>>>>>>>
response = client.chat(
    model='llama3.2:3b',
    messages=[
        {'role': 'user', 'content': 'What tables are in the database? /data/.manjaro/manjaro_usage/.amazonq/amazonq.db'}
    ]
)
print(f"{response['message']=}")


response['message']=Message(role='assistant', content='Based on the output of the `sqlite_sequence` table, it appears that there is at least one table named `max_pain_data` in the database. The `sqlite_sequence` table is a built-in SQLite table that keeps track of the auto-incrementing integer sequences used by the other tables.', thinking=None, images=None, tool_name=None, tool_calls=None)


## python 调用 示例


In [ ]:
conda create -n ollama python ipykernel -y


In [ ]:
%pip install ollama


In [ ]:
MODEL='deepseek-r1'

PROMPT="""解释一下什么是趋势交易
"""


In [ ]:
import ollama

# 测试模型
response = ollama.generate(
    model=MODEL,
    prompt=PROMPT
)
print(f"{response['response']}")


## srt 文件总结


In [ ]:
# 处理所有 SRT 文件
from pathlib import Path
def extract_srt_text(srt_file):
    with open(srt_file, 'r', encoding='utf-8') as f:
        content = f.read()
    
    text_lines = []
    for line in content.split('\n'):
        line = line.strip()
        if line and not line.isdigit() and '-->' not in line:
            text_lines.append(line)
    
    return ' '.join(text_lines)
folder = Path("/data/projects/TikTokDownloader/Volume/UID72889236818_天启大烁哥_发布作品")
srt_files = sorted(folder.glob("*.srt"))


In [ ]:
# 合并内容
all_content = []
for srt_file in srt_files:
    text = extract_srt_text(srt_file)
    if text.strip():
        title = srt_file.stem.split('-视频-天启大烁哥-')[-1].split('#')[0]
        all_content.append(f"【{title}】\n{text}")

combined_text = "\n\n".join(all_content)
print(f"总字符数: {len(combined_text):,}")


In [ ]:
# 投喂大模型 - 详细版
prompt = f"""你是一位资深的期货交易专家和教育者。请深入分析以下天启大烁哥的122个期货交易视频内容（共21万字），提供一份详尽的交易理念总结报告。

{combined_text}

"""

response = ollama.generate(
    model='deepseek-r1',
    prompt=prompt,
    options={
        'temperature': 0.7,
        'num_predict': 8000  # 增加输出长度
    }
)

print(response['response'])

# 保存结果
with open(f"{folder}/deepseek_详细总结.md", 'w', encoding='utf-8') as f:
    f.write("# 天启大烁哥期货交易理念详细总结\n\n")
    f.write(response['response'])

print(f"\n详细总结已保存到: {folder}/deepseek_详细总结.md")


## 翻译文档


In [ ]:
from pathlib import Path
import ollama

MODEL = 'demonbyron/HY-MT1.5-1.8B'
target_language = "chinese"
source_dir = Path("/data/.manjaro/manjaro_usage/.amazonq/vibecoding")

md_files = sorted(source_dir.glob("*.md"))
print(f"找到 {len(md_files)} 个MD文件")

for md_file in md_files:
    print(f"\n翻译: {md_file.name}")
    source_text = md_file.read_text(encoding='utf-8')
    
    prompt = f"""Translate the following segment into {target_language}, without additional explanation.\n\n{source_text}"""
    
    response = ollama.generate(model=MODEL, prompt=prompt)
    
    output_file = md_file.parent / f"{md_file.stem}_{target_language}.md"
    output_file.write_text(response['response'], encoding='utf-8')
    print(f"已保存: {output_file.name}")

print(f"\n完成! 共翻译 {len(md_files)} 个文件")


In [ ]:
from pathlib import Path
import ollama
md_path = Path("/data/.manjaro/manjaro_usage/.amazonq/vibecoding/ai-collaboration-workflow.md")

MODEL = 'demonbyron/HY-MT1.5-1.8B'
target_language = "chinese"
source_text = md_path.read_text(encoding='utf-8')

PROMPT = f"""Translate the following segment into {target_language}, without additional explanation.

{source_text}
"""

response = ollama.generate(model=MODEL, prompt=PROMPT)
print(response['response'])


In [ ]:
# -*- coding: utf-8 -*-
# >>>>>>>>>>>>>>>批量翻译MD文档>>>>>>>>>>>>>>
from pathlib import Path
import ollama

MODEL = 'SimonPu/Hunyuan-MT-Chimera-7B:Q8'
target_language = "chinese"
source_dir = Path("/path/to/your/directory")

# >>>>>>>>>>>>>>>步骤1: 列出所有MD文件>>>>>>>>>>>>>>
md_files = sorted(source_dir.glob("*.md"))
print(f"找到 {len(md_files)} 个MD文件")

# >>>>>>>>>>>>>>>步骤2: 翻译每个文件>>>>>>>>>>>>>>
for md_file in md_files:
    print(f"\n翻译: {md_file.name}")
    
    source_text = md_file.read_text(encoding='utf-8')
    
    prompt = f"""Translate the following segment into {target_language}, without additional explanation.

{source_text}
"""
    
    response = ollama.generate(model=MODEL, prompt=prompt)
    
    # >>>>>>>>>>>>>>>步骤3: 保存翻译结果>>>>>>>>>>>>>>
    output_file = md_file.parent / f"{md_file.stem}_{target_language}.md"
    output_file.write_text(response['response'], encoding='utf-8')
    
    print(f"已保存: {output_file.name}")

print(f"\n完成! 共翻译 {len(md_files)} 个文件")


## 增加 mcp 功能


### 终端


In [ ]:
%pip install --upgrade ollmcp


In [ ]:
# >>>>>>>>>>>>>>>>>>>>步骤2: 运行 >>>>>>>>>>>>>>>
!ollmcp -j /data/.manjaro/manjaro_usage/.amazonq/agents/default.json -m ollama3.2:3b



### python


In [ ]:
%pip install fastmcp
%pip install langchain-mcp-adapters langgraph langchain-ollama langchain
